In [ ]:
%%capture
%pip install websockets

In [1]:
import json
from urllib.parse import quote as url_quote

from requests import delete, get, post
from websockets import ConnectionClosed, connect

In [2]:
response = post("http://localhost:8000/api/v2/agents", json={
    "mode": "conversational",
    "name": "Test Agent 2: examples/create-agent.ipynb",
    "version": "1.0.0",
    "description": "This is a test agent created by the examples/create-agent.ipynb script.",
    "runbook_raw_text": "# Objective\nYou are a helpful assistant.",
    "provider_configs": ["my-openai"],
    "action_packages": [{
        "name": "browsing",
        "version": "1.0.0",
        "organization": "testorg",
        "url": "localhost:31415",
        "api_key": { "value": "some-api-key" },
        "allowed_actions": ["search", "browse"],
    }],
    "agent_architecture": {
        "name": "agent_architecture_default_v2",
        "version": "1.0.0",
    },
    "observability_configs": [],
    "question_groups": [],
    "extra": {
        "test": "test",
    },
})
print(response.json())


{'detail': 'Agent name Test Agent 2: examples/create-agent.ipynb is not unique'}


In [3]:
response = get("http://localhost:8000/api/v2/agents")
print(json.dumps(response.json(), indent=2))

[
  {
    "name": "Test Agent 2: examples/create-agent.ipynb",
    "description": "This is a test agent created by the examples/create-agent.ipynb script.",
    "user_id": "32804463-fd34-4611-a8f6-a756f72840cb",
    "runbook": {
      "raw_text": "# Objective\nYou are a helpful assistant.",
      "content": []
    },
    "version": "1.0.0",
    "provider_configs": [
      "my-openai"
    ],
    "action_packages": [
      {
        "name": "browsing",
        "organization": "testorg",
        "version": "1.0.0",
        "url": "localhost:31415",
        "api_key": {
          "value": "some-api-key"
        },
        "allowed_actions": [
          "search",
          "browse"
        ]
      }
    ],
    "agent_architecture": {
      "name": "agent_architecture_default_v2",
      "version": "1.0.0"
    },
    "question_groups": [],
    "observability_configs": [],
    "created_at": "2025-02-03T22:45:57.572269",
    "updated_at": "2025-02-03T22:45:57.572273",
    "mode": "conversationa

In [4]:
name_url_encoded = url_quote("Test Agent 2: examples/create-agent.ipynb")
agent_by_name = get(f"http://localhost:8000/api/v2/agents/by-name?name={name_url_encoded}").json()
print(json.dumps(agent_by_name, indent=2))

# Get id to use in the next few examples
agent_id = agent_by_name["agent_id"]

{
  "name": "Test Agent 2: examples/create-agent.ipynb",
  "description": "This is a test agent created by the examples/create-agent.ipynb script.",
  "user_id": "32804463-fd34-4611-a8f6-a756f72840cb",
  "runbook": {
    "raw_text": "# Objective\nYou are a helpful assistant.",
    "content": []
  },
  "version": "1.0.0",
  "provider_configs": [
    "my-openai"
  ],
  "action_packages": [
    {
      "name": "browsing",
      "organization": "testorg",
      "version": "1.0.0",
      "url": "localhost:31415",
      "api_key": {
        "value": "some-api-key"
      },
      "allowed_actions": [
        "search",
        "browse"
      ]
    }
  ],
  "agent_architecture": {
    "name": "agent_architecture_default_v2",
    "version": "1.0.0"
  },
  "question_groups": [],
  "observability_configs": [],
  "created_at": "2025-02-03T22:45:57.572269",
  "updated_at": "2025-02-03T22:45:57.572273",
  "mode": "conversational",
  "agent_id": "a6751537-672c-4729-bdc3-1b2a3cc1e0bf",
  "extra": {
   

In [5]:
thread_response = post("http://localhost:8000/api/v2/threads", json={
    "agent_id": agent_id,
    "name": "Test Thread",
    "messages": [
        {
            "role": "user",
            "agent_metadata": {
                "test": "test",
            },
            "server_metadata": {},
            "content": [
                {
                    "kind": "text",
                    "text": "Hello, world!",
                    "citations": [],
                },
            ],
        },
    ],
})
print(json.dumps(thread_response.json(), indent=2))

# Store thread_id for later use
thread_id = thread_response.json()["thread_id"]

{
  "user_id": "32804463-fd34-4611-a8f6-a756f72840cb",
  "agent_id": "a6751537-672c-4729-bdc3-1b2a3cc1e0bf",
  "name": "Test Thread",
  "thread_id": "ee543e05-052e-4661-ab6a-a1ac54b63823",
  "messages": [
    {
      "content": [
        {
          "content_id": "b74d7869-c57d-4dc8-a63b-c4f57f783ef0",
          "kind": "text",
          "text": "Hello, world!",
          "citations": []
        }
      ],
      "role": "user",
      "commited": false,
      "created_at": "2025-02-04T09:33:55.006814",
      "updated_at": "2025-02-04T09:33:55.006816",
      "agent_metadata": {
        "test": "test"
      },
      "server_metadata": {},
      "message_id": "3db0625a-3aff-4c0f-a36f-f57511f50719"
    }
  ],
  "created_at": "2025-02-04T09:33:55.006868",
  "updated_at": "2025-02-04T09:33:55.006869",
  "metadata": {}
}


In [7]:
# Use the agent_id from your earlier create-agent example
uri = f"ws://localhost:8000/api/v2/runs/{agent_id}/stream"

async with connect(uri) as websocket:
    # Send initial payload
    await websocket.send(json.dumps({
        "agent_id": agent_id,
        "thread_id": thread_id,
    }))
    
    # Set up concurrent tasks for sending and receiving
    async def receive_messages():
        while True:
            try:
                message = await websocket.recv()

                # If the message['type'] == 'user_message_request', we need to produce 
                # an input and send the reply back over the websocket.
                message_dict = json.loads(message)
                message_dict = message_dict['event'] if 'event' in message_dict else message_dict
                
                if 'type' not in message_dict:
                    print(f"Unknown message: {json.dumps(json.loads(message), indent=2)}")
                    continue
                
                if message_dict['type'] == 'user_message_request':
                    # Produce an input in the notebook.
                    users_input = input("Enter a reply:>")
                    # Send the reply back over the websocket.
                    await websocket.send(json.dumps({"type": "user_message_reply", "input": users_input}))
                elif message_dict['type'] == 'delta' and message_dict['delta']['event_type'] == 'message_end':
                    for content in message_dict['delta']['data']['content']:
                        if content['kind'] == 'text':
                            print(content['text'])
            except ConnectionClosed:
                print("Connection closed")
                break
    
    # Run both tasks concurrently
    await receive_messages()


Hey! What's your name?
Hey Jordan, where are you from?
Hey Jordan, what's your favorite food?
... TODO!
Connection closed


In [8]:
# Now list threads for the agent
specific_thread = get(f"http://localhost:8000/api/v2/threads?agent_id={agent_id}").json()
print(json.dumps(specific_thread, indent=2))

[
  {
    "user_id": "32804463-fd34-4611-a8f6-a756f72840cb",
    "agent_id": "a6751537-672c-4729-bdc3-1b2a3cc1e0bf",
    "name": "Test Thread",
    "thread_id": "01e0de4c-84e3-4761-ba5f-f0213cb2d3fe",
    "messages": [],
    "created_at": "2025-02-03T22:45:59.186520",
    "updated_at": "2025-02-03T22:45:59.186521",
    "metadata": {}
  },
  {
    "user_id": "32804463-fd34-4611-a8f6-a756f72840cb",
    "agent_id": "a6751537-672c-4729-bdc3-1b2a3cc1e0bf",
    "name": "Test Thread",
    "thread_id": "ee543e05-052e-4661-ab6a-a1ac54b63823",
    "messages": [],
    "created_at": "2025-02-04T09:33:55.006868",
    "updated_at": "2025-02-04T09:33:55.006869",
    "metadata": {}
  }
]


In [9]:
specific_agent = get(f"http://localhost:8000/api/v2/agents/{agent_id}").json()
print(json.dumps(specific_agent, indent=2))

{
  "name": "Test Agent 2: examples/create-agent.ipynb",
  "description": "This is a test agent created by the examples/create-agent.ipynb script.",
  "user_id": "32804463-fd34-4611-a8f6-a756f72840cb",
  "runbook": {
    "raw_text": "# Objective\nYou are a helpful assistant.",
    "content": []
  },
  "version": "1.0.0",
  "provider_configs": [
    "my-openai"
  ],
  "action_packages": [
    {
      "name": "browsing",
      "organization": "testorg",
      "version": "1.0.0",
      "url": "localhost:31415",
      "api_key": {
        "value": "some-api-key"
      },
      "allowed_actions": [
        "search",
        "browse"
      ]
    }
  ],
  "agent_architecture": {
    "name": "agent_architecture_default_v2",
    "version": "1.0.0"
  },
  "question_groups": [],
  "observability_configs": [],
  "created_at": "2025-02-03T22:45:57.572269",
  "updated_at": "2025-02-03T22:45:57.572273",
  "mode": "conversational",
  "agent_id": "a6751537-672c-4729-bdc3-1b2a3cc1e0bf",
  "extra": {
   

In [10]:
delete(f"http://localhost:8000/api/v2/agents/{agent_id}")

<Response [204]>

In [11]:
# If we get the agent again, it should be gone
specific_agent = get(f"http://localhost:8000/api/v2/agents/{agent_id}")
print(json.dumps({
    "status_code": specific_agent.status_code,
    "response": specific_agent.json(),
}, indent=2))


{
  "status_code": 404,
  "response": {
    "detail": "Agent a6751537-672c-4729-bdc3-1b2a3cc1e0bf not found"
  }
}
